# 05 — Actions Training Pairs

ARO's action vocabulary is the model's building blocks — get it wrong and every
generated program breaks. This notebook makes sure the model knows each action cold.

For every action in `list-of-actions.txt`, we generate multiple training pair types:

1. **Usage** — "Show me how to use `retrieve`" → the canonical example line
2. **Alias** — alternate verbs that map to the same action (only real ARO verbs)
3. **Explanation** — "What does this statement do?" → role + preposition summary
4. **Which action** — "Which action retrieves from a repository?" → name + example
5. **In context** — a minimal Application-Start that uses the action (teaches structure)
6. **LLM-generated** — fuller feature sets, validated with `aro check`

We also add a set of static **error-pattern pairs**: every common ARO mistake paired
with the correct fix. Seeing what *not* to do is surprisingly effective training signal.

**Input:**  `../../Examples/list-of-actions.txt`
            `../data/02_knowledge/knowledge.json`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)
            `../data/04b_actions/actions_pairs.jsonl` (own copy)

In [1]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess
from pathlib import Path

ACTIONS_FILE = ARO_ROOT / 'Examples' / 'list-of-actions.txt'
DATA_OUT     = DATA_ROOT / '04b_actions'
DATA_OUT.mkdir(parents=True, exist_ok=True)

OWN_FILE   = DATA_OUT / 'actions_pairs.jsonl'

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

print(f'Actions file: {ACTIONS_FILE}')
print(f'Knowledge base: {len(kb["actions"])} actions')

TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO/ARO-Train
Pairs:     /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters
Actions file: /Users/kris/Projects/ARO/ARO-Train/Examples/list-of-actions.txt
Knowledge base: 54 actions


## Parse list-of-actions.txt

In [2]:
def parse_actions_file(path):
    """
    Parses list-of-actions.txt into a list of dicts:
      { verb, role, example_line, category }
    Sections are delimited by lines containing '---'.
    """
    content = Path(path).read_text()
    lines   = content.splitlines()

    actions     = []
    current_role = 'UNKNOWN'
    role_re      = re.compile(r'^(\w+) actions', re.IGNORECASE)
    # Match lines like: Extract the <id> from the <pathParameters: id>.
    stmt_re      = re.compile(r'^([A-Z][a-zA-Z]+)\s+(?:the|a|an)?\s*<')

    for line in lines:
        stripped = line.strip()
        # Section header — detect role
        role_m = role_re.match(stripped)
        if role_m:
            current_role = role_m.group(1).upper()
            continue
        # Skip separators and headers
        if not stripped or stripped.startswith('ARO Action') or set(stripped) == {'-'}:
            continue
        # Match statement lines
        stmt_m = stmt_re.match(stripped)
        if stmt_m:
            verb = stmt_m.group(1).lower()
            actions.append({
                'verb':         verb,
                'role':         current_role,
                'example_line': stripped,
            })

    return actions

raw_actions = parse_actions_file(ACTIONS_FILE)
print(f'Parsed {len(raw_actions)} action examples')
for a in raw_actions[:5]:
    print(f'  [{a["role"]:10s}] {a["verb"]:12s} → {a["example_line"][:60]}')

Parsed 58 action examples
  [REQUEST   ] extract      → Extract the <id> from the <pathParameters: id>.
  [REQUEST   ] retrieve     → Retrieve the <user> from the <user-repository> where id = <u
  [REQUEST   ] receive      → Receive the <packet> from the <socket: message>.
  [REQUEST   ] read         → Read the <content> from the <config-file>.
  [REQUEST   ] request      → Request the <response> from the <api-url>.


## Build action metadata index

Merge parsed examples with metadata from `knowledge.json` (verbs, prepositions, description).

In [3]:
# Build a lookup: primary_verb -> action metadata
action_meta = {}
for a in kb.get('actions', []):
    for v in a.get('verbs', []):
        action_meta[v.lower()] = a

# Enrich parsed actions with metadata
for a in raw_actions:
    meta = action_meta.get(a['verb'], {})
    a['all_verbs']     = meta.get('verbs', [a['verb']])
    a['prepositions']  = meta.get('prepositions', [])
    a['description']   = meta.get('description', '')
    a['action_name']   = meta.get('name', a['verb'].capitalize() + 'Action')

print(f'Enriched {len(raw_actions)} actions with metadata')
print(f'Actions with description: {sum(1 for a in raw_actions if a["description"])}')

Enriched 58 actions with metadata
Actions with description: 51


## Load model

In [4]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

model, tokenizer, _, mlx_generate, make_sampler = load_model()

TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO/ARO-Train
Pairs:     /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters


/Users/kris/Projects/ARO/ARO-Train/Train/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit with warm-start adapter...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 202950.19it/s]


  Adapter: /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters/warm_start
Model ready.


## System prompt

In [5]:
syntax_summary = kb.get('aro_syntax', '')[:2000]

SYSTEM_PROMPT = f"""You are an expert ARO (Action Result Object) programmer and teacher.

ARO syntax: Verb the <Result> preposition [the] <Object>.
Every statement is either:
  Verb the <Result: qualifier> from/to/with/for/on/into the <Object: qualifier>.
  Verb <literal> to the <Object>.
  Publish as <alias> <variable>.

{syntax_summary}

Rules:
- Variables are immutable — use new names for each transform
- String concat: ++ (not +)
- Feature set: (Name: Business Activity) {{ statements }}
- Exactly one Application-Start per application
- Return an <OK: status> ... to end a feature set
- Qualifiers after : in angle brackets specify field/operation"""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

System prompt: 2618 chars


## Pair templates

For each action, generate 5 types of static training pairs:

1. **Usage** — "Show me how to use `retrieve`" → wrapped in feature set (not bare one-liner)
2. **Alias** — alternate verbs that map to the same action (with explanation + wrapped example)
3. **Explanation** — "What does this statement do?" → role, verbs, prepositions (derived from example line if metadata missing)
4. **Which action** — "Which action retrieves from a repository?" → name + fenced example
5. **In context** — a minimal Application-Start using the action (teaches structure)

In [6]:
# Known valid ARO verbs (from list-of-actions.txt) — alias pairs are only generated
# when the alias is itself a recognized ARO verb, not an internal implementation detail.
VALID_ARO_VERBS = {
    'extract', 'retrieve', 'receive', 'read', 'request', 'list', 'stat', 'exists',
    'prompt', 'select', 'stream', 'compute', 'validate', 'compare', 'transform',
    'create', 'update', 'sort', 'merge', 'delete', 'map', 'reduce', 'filter',
    'call', 'execute', 'parse', 'accept', 'join', 'split', 'sleep', 'return',
    'throw', 'store', 'log', 'send', 'emit', 'notify', 'write', 'append',
    'schedule', 'make', 'copy', 'move', 'clear', 'show', 'render', 'repaint',
    'start', 'stop', 'listen', 'keepalive', 'connect', 'broadcast', 'close',
    'given', 'when', 'then', 'assert', 'fetch', 'get', 'put', 'post', 'patch',
    'calculate', 'derive', 'invoke', 'run', 'shell', 'publish', 'push',
}

# ARO prepositions for extraction from example lines
_ARO_PREPOSITIONS = {'from', 'to', 'with', 'for', 'on', 'into', 'at', 'by',
                     'where', 'against', 'via', 'using', 'in'}

def _extract_prepositions_from_line(line):
    """Extract prepositions actually used in an example ARO statement."""
    words = line.lower().replace('.', '').split()
    return [w for w in words if w in _ARO_PREPOSITIONS]


def make_static_pairs(action):
    """
    Returns static (non-generated) pairs that don't require the LLM.
    These are high-quality, deterministic pairs.
    """
    verb   = action['verb']
    line   = action['example_line']
    role   = action['role']
    verbs  = action['all_verbs']
    preps  = action['prepositions']
    name   = action['action_name']

    # If metadata has no prepositions, derive from the example line
    if not preps:
        preps = _extract_prepositions_from_line(line)

    verb_list  = ', '.join(f'`{v}`' for v in verbs[:4])
    prep_list  = ', '.join(f'`{p}`' for p in preps[:4]) if preps else '(none documented)'

    pairs = []

    # 1. Direct usage — wrap in a feature set so the model learns structure
    pairs.append({
        'instruction': f'Show me how to use the `{verb}` action in ARO.',
        'output':      (
            f'```aro\n'
            f'(Application-Start: Demo) {{\n'
            f'    {line}\n'
            f'    Return an <OK: status> for the <result>.\n'
            f'}}\n'
            f'```'
        ),
        'source':      'actions_usage',
        'task_type':   'usage',
        'score':       1.0,
    })

    # 2. Verb alias questions — only for aliases that are real ARO verbs
    for alias in verbs[1:4]:
        if alias.lower() not in VALID_ARO_VERBS:
            continue
        rewritten = line.replace(verb.capitalize(), alias.capitalize(), 1)
        if rewritten != line and re.match(r'^[A-Z][a-z]+\s+', rewritten):
            pairs.append({
                'instruction': f'How do I use `{alias}` in ARO? (alias for {verb})',
                'output':      (
                    f'`{alias.capitalize()}` is an alias for the `{verb}` action. Example:\n\n'
                    f'```aro\n'
                    f'(Application-Start: Demo) {{\n'
                    f'    {rewritten}\n'
                    f'    Return an <OK: status> for the <result>.\n'
                    f'}}\n'
                    f'```'
                ),
                'source':      'actions_alias',
                'task_type':   'alias',
                'score':       0.9,
            })

    # 3. What does this line do?
    pairs.append({
        'instruction': f'What does this ARO statement do?\n\n{line}',
        'output':      (
            f'This statement uses the `{verb}` action ({role} role).\n'
            f'Available verbs: {verb_list}\n'
            f'Valid prepositions: {prep_list}\n\n'
            f'The result is bound to the variable in the first angle bracket '
            f'(`<result>`), and the data source is in the second angle bracket '
            f'(`<object>`). The preposition `{preps[0] if preps else "from"}` '
            f'connects them.'
        ),
        'source':      'actions_explain',
        'task_type':   'explain',
        'score':       1.0,
    })

    # 4. Which action to use?
    role_desc = {
        'REQUEST':  'pulls data from an external source into the feature set',
        'OWN':      'transforms or computes data within the feature set',
        'RESPONSE': 'returns a result or error to the caller',
        'EXPORT':   'persists, broadcasts, or publishes data',
        'FILE':     'performs a file system operation',
        'TERMINAL': 'interacts with the terminal display',
        'SERVICE':  'manages a long-running service',
        'TEST':     'performs a test assertion',
        'SCHEDULE': 'schedules a recurring event',
    }.get(role, 'performs an operation')

    pairs.append({
        'instruction': f'In ARO, which action {role_desc} using the verb `{verb}`? Give an example.',
        'output':      (
            f'The `{verb}` action ({name}).\n\n'
            f'Example:\n'
            f'```aro\n{line}\n```'
        ),
        'source':      'actions_which',
        'task_type':   'which_action',
        'score':       1.0,
    })

    # 5. Show in context — a mini feature set
    pairs.append({
        'instruction': f'Write a minimal ARO Application-Start feature set that uses the `{verb}` action.',
        'output':      (
            f'```aro\n'
            f'(Application-Start: Demo) {{\n'
            f'    {line}\n'
            f'    Return an <OK: status> for the <result>.\n'
            f'}}\n'
            f'```'
        ),
        'source':      'actions_context_static',
        'task_type':   'context_static',
        'score':       1.0,
    })

    return pairs


# Test
sample = raw_actions[0]
static_pairs = make_static_pairs(sample)
print(f'Static pairs for "{sample["verb"]}": {len(static_pairs)}')
for p in static_pairs:
    print(f'  [{p["source"]}] {p["instruction"][:60]}')
# Verify prepositions are now populated
print(f'\nExtract prepositions check:')
for p in static_pairs:
    if p['source'] == 'actions_explain':
        print(f'  {p["output"][:200]}')

Static pairs for "extract": 6
  [actions_usage] Show me how to use the `extract` action in ARO.
  [actions_alias] How do I use `parse` in ARO? (alias for extract)
  [actions_alias] How do I use `get` in ARO? (alias for extract)
  [actions_explain] What does this ARO statement do?

Extract the <id> from the 
  [actions_which] In ARO, which action pulls data from an external source into
  [actions_context_static] Write a minimal ARO Application-Start feature set that uses 

Extract prepositions check:
  This statement uses the `extract` action (REQUEST role).
Available verbs: `extract`, `parse`, `get`
Valid prepositions: `from`, `via`

The result is bound to the variable in the first angle bracket (`


## LLM-generated pairs

For each action, ask the model to write complete feature sets using the action.
Generated code goes through a 3-layer validation pipeline:
1. Try raw `aro check`
2. If bare snippet → auto-wrap in feature set and retry
3. If still failing → feed error back to LLM for repair (2 attempts)

All outputs are wrapped in `` ```aro `` fences for consistency.

In [7]:
import subprocess, tempfile

def aro_check_code(code_or_block):
    """Run aro check on a string. Returns (True/False/None, error_str)."""
    # Strip markdown fences if present
    code = re.sub(r'^```[a-z]*\n?', '', code_or_block.strip())
    code = re.sub(r'\n?```$', '', code.strip())
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=8)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:500]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def auto_wrap_aro(code):
    """Wrap bare ARO statements in a feature set if they don't have one."""
    stripped = code.strip()
    if stripped.startswith('(') and '{' in stripped.split('\n')[0]:
        return code, False
    if '<statements>' in stripped or '^^^^' in stripped or '❌' in stripped:
        return None, False
    if '//' in stripped and '(*' not in stripped:
        return None, False
    has_aro = any(re.search(r'<[a-z][-a-z0-9]*', line)
                  for line in stripped.split('\n')
                  if not line.strip().startswith('(*'))
    if not has_aro:
        return None, False
    has_return = any(re.match(r'\s*(Return|Throw)\b', line)
                     for line in stripped.split('\n'))
    indented = '\n'.join(f'    {line}' for line in stripped.split('\n'))
    if has_return:
        wrapped = f'(Application-Start: Example) {{\n{indented}\n}}'
    else:
        wrapped = f'(Application-Start: Example) {{\n{indented}\n    Return an <OK: status> for the <result>.\n}}'
    return wrapped, True


MAX_NEW_TOKENS = 600

def generate(prompt, temp=0.5):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    out    = mlx_generate(
        model, tokenizer,
        prompt=tokens,
        max_tokens=MAX_NEW_TOKENS,
        sampler=make_sampler(temp=temp),
        verbose=False,
    )
    return out.strip()


def extract_aro_blocks(text):
    """Extract ARO code blocks from markdown-fenced text."""
    blocks = re.findall(r'```aro\n(.*?)```', text, re.DOTALL)
    return [b.strip() for b in blocks if b.strip()]


def repair_aro(code, error, prompt, max_retries=2):
    """Feed aro check error back to LLM for repair."""
    current = code
    for attempt in range(max_retries):
        repair_prompt = (
            f'{prompt}\n\n'
            f'My previous attempt:\n```aro\n{current}\n```\n\n'
            f'This failed aro check with:\n{error}\n\n'
            f'Fix the code. Reply with ONLY corrected code in ```aro ... ``` fences.'
        )
        fixed_text = generate(repair_prompt)
        blocks = extract_aro_blocks(fixed_text)
        if not blocks:
            return None, False, attempt + 1
        current = blocks[0]
        ok, err = aro_check_code(current)
        if ok is True:
            return current, True, attempt + 1
        if ok is None:
            return current, None, attempt + 1
        error = err
    return None, False, max_retries


# Track LLM generation stats
llm_stats = {'attempted': 0, 'valid_first': 0, 'wrapped': 0, 'repaired': 0, 'dropped': 0}

CONTEXT_PROMPTS = [
    lambda a: (
        f"Write a complete ARO application (Application-Start + one handler) "
        f"that demonstrates the `{a['verb']}` action.\n"
        f"Example of the action: `{a['example_line']}`\n"
        f"Use a realistic domain (e.g. user management, e-commerce, file processing).\n"
        f"The code MUST be wrapped in ```aro ... ``` fences and pass `aro check`."
    ),
    lambda a: (
        f"Write an ARO feature set that uses `{a['verb']}` together with at least "
        f"two other related actions. Show the full context with Application-Start.\n"
        f"The code MUST be wrapped in ```aro ... ``` fences and pass `aro check`."
    ),
]

def make_llm_pairs(action):
    """Generate LLM context pairs, with auto-wrap and repair fallbacks."""
    pairs = []
    for prompt_fn in CONTEXT_PROMPTS:
        prompt = prompt_fn(action)
        llm_stats['attempted'] += 1
        try:
            raw_output = generate(prompt)
            if len(raw_output.strip()) < 20:
                llm_stats['dropped'] += 1
                continue

            # Extract code from fences if present, otherwise treat as raw code
            blocks = extract_aro_blocks(raw_output)
            if blocks:
                code = blocks[0]
            else:
                # LLM didn't use fences — strip any partial fences and use raw
                code = re.sub(r'^```[a-z]*\n?', '', raw_output.strip())
                code = re.sub(r'\n?```$', '', code.strip())

            # Validate
            check_ok, check_err = aro_check_code(code)

            # Auto-wrap bare snippets
            if check_ok is False:
                wrap_result, was_wrapped = auto_wrap_aro(code)
                if wrap_result is not None and was_wrapped:
                    wrap_ok, _ = aro_check_code(wrap_result)
                    if wrap_ok is True:
                        code = wrap_result
                        check_ok = True
                        llm_stats['wrapped'] += 1

            # Repair loop
            if check_ok is False:
                fixed, check_ok, attempts = repair_aro(code, check_err, prompt)
                if check_ok is True:
                    code = fixed
                    llm_stats['repaired'] += 1
                    print(f'    ✓ {action["verb"]}: repaired in {attempts} attempt(s)', flush=True)
                else:
                    llm_stats['dropped'] += 1
                    short_err = check_err.split('\n')[0][:60] if check_err else '?'
                    print(f'    x {action["verb"]}: {short_err}', flush=True)
                    continue

            if check_ok is True:
                llm_stats['valid_first'] += 1

            score = 1.0 if check_ok is True else 0.85
            # Always wrap output in aro fences for consistency
            output = f'```aro\n{code}\n```'
            pairs.append({
                'instruction': prompt,
                'output':      output,
                'source':      'actions_context',
                'task_type':   'context_llm',
                'score':       score,
            })
        except Exception as e:
            llm_stats['dropped'] += 1
            print(f'  [LLM] Error for {action["verb"]}: {e}')
    return pairs

print('LLM pair generation ready (with auto-wrap + repair loop).')

LLM pair generation ready (with auto-wrap + repair loop).


## Main generation loop

In [8]:
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# source → task_type mapping for backfilling resumed pairs
_SOURCE_TO_TASK = {
    'actions_usage':          'usage',
    'actions_alias':          'alias',
    'actions_explain':        'explain',
    'actions_which':          'which_action',
    'actions_context_static': 'context_static',
    'actions_context':        'context_llm',
    'error_pattern':          'error_pattern',
}

all_pairs = []

# Resume: load already-generated pairs
already_verbs = set()
if OWN_FILE.exists():
    with open(OWN_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                # Backfill task_type on old pairs that lack it
                if 'task_type' not in p:
                    p['task_type'] = _SOURCE_TO_TASK.get(p.get('source', ''), 'unknown')
                all_pairs.append(p)
    # Reconstruct which verbs are done from static pairs
    for p in all_pairs:
        m = re.search(r'`(\w+)` action', p.get('instruction', ''))
        if m:
            already_verbs.add(m.group(1).lower())
    print(f'Resuming — {len(all_pairs)} pairs already generated')

outf = open(OWN_FILE, 'a')

try:
    pbar = tqdm(total=len(raw_actions), desc='Actions', unit='action')
    for action in raw_actions:
        verb = action['verb']
        pbar.set_description(f'Actions [{verb}]')

        # ── Static pairs (always regenerate to keep idempotent) ──────────────
        if verb not in already_verbs:
            static = make_static_pairs(action)
            for p in static:
                all_pairs.append(p)
                outf.write(json.dumps(p) + '\n')

            # ── LLM-generated context pairs ──────────────────────────────────
            llm = make_llm_pairs(action)
            for p in llm:
                all_pairs.append(p)
                outf.write(json.dumps(p) + '\n')

            already_verbs.add(verb)
            outf.flush()
            pbar.set_postfix({'total': len(all_pairs), 'verb': verb})

        pbar.update(1)
finally:
    pbar.close()
    outf.close()

print(f'\nTotal action training pairs: {len(all_pairs)}')
from collections import Counter
source_counts = Counter(p.get('source', '') for p in all_pairs)
for src, cnt in sorted(source_counts.items()):
    print(f'  {src:<25} {cnt:4d}')

Resuming — 325 pairs already generated


Actions [assert]: 100%|██████████| 58/58 [00:00<00:00, 6897.55action/s]   


Total action training pairs: 325
  actions_alias                9
  actions_context             84
  actions_context_static      58
  actions_explain             58
  actions_usage               58
  actions_which               58


## Merge into main knowledge pairs

In [9]:
clean_notebook_pairs('NB05')

new_count = 0
pairs_to_save = []
for pair in all_pairs:
    pair['notebook'] = 'NB05'
    pairs_to_save.append(pair)

new_count = save_notebook_pairs('NB05', pairs_to_save)

print(f'Merged {new_count} new action pairs into {PAIRS_FILE}')

# Print LLM generation stats
print(f'\nLLM generation stats:')
print(f'  Attempted:    {llm_stats["attempted"]}')
print(f'  Valid (1st):  {llm_stats["valid_first"]}')
print(f'  Auto-wrapped: {llm_stats["wrapped"]}')
print(f'  Repaired:     {llm_stats["repaired"]}')
print(f'  Dropped:      {llm_stats["dropped"]}')

print()
print('Next steps:')
print('  → Run notebook 04 again to retrain warm-start adapter on enriched pairs')
print('  → Or proceed to notebook 06 (execution-grounded pairs)')

[NB05] Cleaned 341 pairs from previous run
Merged 325 new action pairs into /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl

LLM generation stats:
  Attempted:    0
  Valid (1st):  0
  Auto-wrapped: 0
  Repaired:     0
  Dropped:      0

Next steps:
  → Run notebook 04 again to retrain warm-start adapter on enriched pairs
  → Or proceed to notebook 06 (execution-grounded pairs)


## Static error-pattern pairs (no LLM needed)

Every common ARO mistake paired with the correct fix.
These are deterministic — no generation, always correct — and teach the model
exactly what NOT to do, which improves generation quality more than positive
examples alone.

In [10]:
ERROR_PATTERNS = [
    # ── String concatenation ──────────────────────────────────────────────────
    {
        'instruction': 'Fix the bug in this ARO code:\n\nCompute the <greeting> from "Hello, " + <name>.',
        'output': (
            'The `+` operator is not used for string concatenation in ARO. '
            'Use `++` instead:\n\n'
            '```aro\nCompute the <greeting> from "Hello, " ++ <name>.\n```'
        ),
    },
    {
        'instruction': 'What is wrong with: `Compute the <msg> from <first> + " " + <last>.`',
        'output': (
            'ARO uses `++` for string concatenation, not `+`. '
            'The `+` operator is only for numeric addition. Correct version:\n\n'
            '```aro\nCompute the <msg> from <first> ++ " " ++ <last>.\n```'
        ),
    },

    # ── Reserved variable name prefixes ──────────────────────────────────────
    {
        'instruction': 'Fix the bug: `Compute the <is-valid> from <age> > 18.`',
        'output': (
            '`is-` is a reserved prefix in ARO and cannot start a variable name. '
            'Rename the variable:\n\n'
            '```aro\nCompute the <valid> from <age> > 18.\n```'
        ),
    },
    {
        'instruction': 'Why does `Create the <with-data> with <payload>.` fail in ARO?',
        'output': (
            '`with-` is a reserved prefix in ARO because `with` is a preposition keyword. '
            'Use a different name:\n\n'
            '```aro\nCreate the <data> with <payload>.\n```'
        ),
    },
    {
        'instruction': 'Fix the variable naming error: `Compute the <empty-result> from <list>.`',
        'output': (
            '`empty-` is a reserved prefix in ARO. Choose a different variable name:\n\n'
            '```aro\nCompute the <blank-result> from <list>.\n```'
        ),
    },

    # ── Missing Application-Start ─────────────────────────────────────────────
    {
        'instruction': (
            'Fix the ARO application structure — it gives "no Application-Start found":\n\n'
            '```aro\n(listUsers: User API) {\n    Retrieve the <users> from the <user-repository>.\n'
            '    Return an <OK: status> with <users>.\n}\n```'
        ),
        'output': (
            'Every ARO application requires exactly one `Application-Start` feature set. '
            'Add it along with the HTTP server startup:\n\n'
            '```aro\n(Application-Start: User Service) {\n'
            '    Start the <http-server> with <contract>.\n'
            '    Keepalive the <application> for the <events>.\n'
            '    Return an <OK: status> for the <startup>.\n}\n\n'
            '(listUsers: User API) {\n'
            '    Retrieve the <users> from the <user-repository>.\n'
            '    Return an <OK: status> with <users>.\n}\n```'
        ),
    },

    # ── Wrong preposition ─────────────────────────────────────────────────────
    {
        'instruction': 'Fix: `Retrieve the <user> with the <user-repository>.`',
        'output': (
            'The `Retrieve` action uses the `from` preposition when reading from a repository, '
            'not `with`. `with` is used to pass data TO an action. Correct:\n\n'
            '```aro\nRetrieve the <user> from the <user-repository>.\n```'
        ),
    },
    {
        'instruction': 'Fix: `Store the <user> from the <user-repository>.`',
        'output': (
            'The `Store` action uses `into` (or `in`) to specify the target repository, '
            'not `from`. `from` is for reading. Correct:\n\n'
            '```aro\nStore the <user> into the <user-repository>.\n```'
        ),
    },
    {
        'instruction': 'Fix: `Return a <Created: status> for the <user>` on a POST endpoint.',
        'output': (
            'The `Return` action for a successful resource creation uses `with` to attach '
            'the created resource, and `for` for context. For POST endpoints:\n\n'
            '```aro\nReturn a <Created: status> with <user>.\n```'
        ),
    },

    # ── Emit syntax ───────────────────────────────────────────────────────────
    {
        'instruction': 'Fix: `Emit the <user> to the <user-created-event>.`',
        'output': (
            'Emit requires the event type in result position with `: event` qualifier '
            'and the payload after `with`. This example puts data in result position, '
            'event in object, and uses wrong preposition `to`. Correct:\n\n'
            '```aro\nEmit a <UserCreated: event> with <user>.\n```'
        ),
    },

    # ── Log syntax ────────────────────────────────────────────────────────────
    {
        'instruction': 'Fix: `Log the <message> for the <console>.`',
        'output': (
            'Log uses the preposition `to`, not `for`. Correct:\n\n'
            '```aro\nLog <message> to the <console>.\n```'
        ),
    },

    # ── openapi.yaml / operationId mismatch ──────────────────────────────────
    {
        'instruction': (
            'The HTTP route defined in openapi.yaml as operationId `getUser` is never called. '
            'This ARO code exists but is not triggered:\n\n'
            '```aro\n(Get User: User API) {\n    Extract the <id> from the <pathParameters: id>.\n'
            '    Retrieve the <user> from the <user-repository> where id = <id>.\n'
            '    Return an <OK: status> with <user>.\n}\n```'
        ),
        'output': (
            'The feature set name must exactly match the `operationId` in openapi.yaml. '
            '`Get User` (with space) does not match `getUser` (camelCase). Fix:\n\n'
            '```aro\n(getUser: User API) {\n    Extract the <id> from the <pathParameters: id>.\n'
            '    Retrieve the <user> from the <user-repository> where id = <id>.\n'
            '    Return an <OK: status> with <user>.\n}\n```'
        ),
    },

    # ── Immutability / rebinding ──────────────────────────────────────────────
    {
        'instruction': (
            'Why does this fail in ARO?\n\n'
            '```aro\nCreate the <count> with 0.\nCompute the <count> from <count> + 1.\n```'
        ),
        'output': (
            'Variables in ARO are immutable — you cannot rebind `<count>` once set. '
            'Use a new name for each transformation:\n\n'
            '```aro\nCreate the <initial-count> with 0.\n'
            'Compute the <incremented-count> from <initial-count> + 1.\n```'
        ),
    },

    # ── HTTP server without openapi.yaml ─────────────────────────────────────
    {
        'instruction': 'Why does `Start the <http-server> with <contract>.` fail with "no contract"?',
        'output': (
            'ARO uses contract-first HTTP development. The `<contract>` variable refers to an '
            '`openapi.yaml` file in the application directory. Without this file, the HTTP server '
            'cannot start. Create an `openapi.yaml` in your app directory defining your API paths '
            'and operationIds, then the contract will be auto-loaded.'
        ),
    },

    # ── Path parameter extraction ─────────────────────────────────────────────
    {
        'instruction': 'Fix: `Extract the <id> from the <request: id>.` for a URL path parameter.',
        'output': (
            'URL path parameters in ARO are extracted from `pathParameters`, not `request`. '
            '`request` is for the request body. Correct syntax:\n\n'
            '```aro\nExtract the <id> from the <pathParameters: id>.\n```'
        ),
    },

    # ── Keepalive placement ───────────────────────────────────────────────────
    {
        'instruction': (
            'My HTTP server starts but exits immediately. '
            'How do I keep it running?\n\n'
            '```aro\n(Application-Start: My API) {\n'
            '    Start the <http-server> with <contract>.\n'
            '    Return an <OK: status> for the <startup>.\n}\n```'
        ),
        'output': (
            'Add `Keepalive` after starting the server. Without it, the application '
            'exits after Application-Start completes:\n\n'
            '```aro\n(Application-Start: My API) {\n'
            '    Start the <http-server> with <contract>.\n'
            '    Keepalive the <application> for the <events>.\n'
            '    Return an <OK: status> for the <startup>.\n}\n```'
        ),
    },
]

# Write error pattern pairs
error_pairs = []
for pair in ERROR_PATTERNS:
    error_pairs.append({**pair, 'source': 'error_pattern', 'task_type': 'error_pattern', 'score': 1.0, 'notebook': 'NB05'})

error_count = save_notebook_pairs('NB05', error_pairs)

print(f'Added {error_count} static error-pattern pairs to {PAIRS_FILE}')

Added 16 static error-pattern pairs to /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl


In [11]:
import csv
import re as _re_csv
from pathlib import Path
from collections import Counter
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_path = _run_dir / '05_actions_training.csv'

# Build per-verb, per-source counts from all_pairs
_verb_task = Counter()
for _p in all_pairs:
    _m = _re_csv.search(r'`(\w+)`', _p.get('instruction', ''))
    _v = _m.group(1) if _m else 'other'
    _s = _p.get('source', 'unknown')
    _verb_task[(_v, _s)] += 1
# Add error_pattern pairs
_verb_task[('various', 'error_pattern')] = len(ERROR_PATTERNS)

with open(_csv_path, 'w', newline='') as _f:
    _w = csv.writer(_f)
    _w.writerow(['verb', 'task_template', 'pair_count'])
    for (_v, _s), _cnt in sorted(_verb_task.items()):
        _w.writerow([_v, _s, _cnt])

print(f'CSV saved: {_csv_path}  ({len(_verb_task)} rows)')

# Summary
_by_source = Counter(_p.get('source', '') for _p in all_pairs)
_by_source['error_pattern'] = len(ERROR_PATTERNS)
print('\nPairs by category:')
for _s, _n in sorted(_by_source.items(), key=lambda x: -x[1]):
    print(f'  {_s:<25} {_n:4d}')

CSV saved: run/2026-04-28/05_actions_training.csv  (236 rows)

Pairs by category:
  actions_context             84
  actions_usage               58
  actions_explain             58
  actions_which               58
  actions_context_static      58
  error_pattern               16
  actions_alias                9


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
import json
from collections import Counter
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '05_actions_training.png'

# Count pairs by source (always populated, unlike task_type which may be missing on resumed pairs)
_src_counts = Counter(p.get('source', 'unknown') for p in all_pairs)
_src_counts['error_pattern'] = len(ERROR_PATTERNS)

_grand = sum(_src_counts.values())
_n_verbs = len({a['verb'] for a in raw_actions})

# ── Two-panel figure: category bars + LLM funnel ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={'width_ratios': [3, 2]})

# Left: category bar chart (keyed by source)
_src_order  = ['actions_usage', 'actions_alias', 'actions_explain',
               'actions_which', 'actions_context_static', 'actions_context', 'error_pattern']
_src_labels = ['Usage', 'Alias', 'Explain', 'Which action', 'In context\n(static)', 'Context\n(LLM)', 'Error\npatterns']
_src_colors = ['#3498db', '#9b59b6', '#1abc9c', '#e67e22', '#27ae60', '#2980b9', '#e74c3c']

_heights = [_src_counts.get(s, 0) for s in _src_order]

ax = axes[0]
bars = ax.bar(_src_labels, _heights, color=_src_colors, edgecolor='white', width=0.65)
for bar, h in zip(bars, _heights):
    if h:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                str(h), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Training pairs')
ax.set_title(f'Actions Training — {_grand:,} pairs across {_n_verbs} verbs',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(_heights, default=1) * 1.25)
ax.grid(axis='y', alpha=0.3)

# Right: LLM generation funnel
ax2 = axes[1]
_funnel_labels = ['Valid\n(1st try)', 'Auto-\nwrapped', 'Repaired', 'Dropped']
_funnel_values = [
    llm_stats.get('valid_first', 0),
    llm_stats.get('wrapped', 0),
    llm_stats.get('repaired', 0),
    llm_stats.get('dropped', 0),
]
_funnel_colors = ['#2ecc71', '#1abc9c', '#3498db', '#e74c3c']

_total_llm = llm_stats.get('attempted', 0)
_valid_llm = _funnel_values[0] + _funnel_values[1] + _funnel_values[2]

if _total_llm > 0:
    bars2 = ax2.bar(_funnel_labels, _funnel_values, color=_funnel_colors, edgecolor='white', width=0.6)
    for bar, h in zip(bars2, _funnel_values):
        if h:
            ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                    str(h), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2.set_ylabel('LLM attempts')
    ax2.set_title(f'LLM Context Generation — {_valid_llm}/{_total_llm} '
                  f'({100*_valid_llm/max(_total_llm,1):.0f}% yield)',
                  fontsize=12, fontweight='bold')
    ax2.set_ylim(0, max(_funnel_values, default=1) * 1.3)
else:
    # All LLM pairs were resumed — show summary instead
    _ctx_count = _src_counts.get('actions_context', 0)
    ax2.text(0.5, 0.5,
             f'LLM context pairs: {_ctx_count}\n(resumed from previous run)\n\n'
             f'Delete {OWN_FILE.name}\nto regenerate with\nauto-wrap + repair',
             transform=ax2.transAxes, ha='center', va='center',
             fontsize=11, color='#555', style='italic')
    ax2.set_title('LLM Context Generation — resumed', fontsize=12, fontweight='bold')
    ax2.set_xticks([])
    ax2.set_yticks([])
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

Saved: run/2026-04-28/05_actions_training.png


In [13]:
# ── Samples per category ─────────────────────────────────────────────────────
import json, random
from collections import defaultdict

_pairs = []
if OWN_FILE.exists():
    with open(OWN_FILE) as f:
        for line in f:
            if line.strip():
                _pairs.append(json.loads(line))

_by_src = defaultdict(list)
for p in _pairs:
    _by_src[p.get('source', 'unknown')].append(p)

_CATEGORY_ORDER = [
    'actions_usage',
    'actions_alias',
    'actions_explain',
    'actions_which',
    'actions_context_static',
    'actions_context',
    'error_pattern',
]
SAMPLES_PER_CAT = 2

for src in _CATEGORY_ORDER:
    pool = _by_src.get(src, [])
    if not pool:
        continue
    print(f'\n{"─"*72}')
    print(f'  {src}  ({len(pool)} pairs)')
    print('─'*72)
    for s in random.sample(pool, min(SAMPLES_PER_CAT, len(pool))):
        print(f'Q: {s["instruction"][:220]}')
        out = s.get('output', '')
        print(f'A: {out[:320]}{"..." if len(out) > 320 else ""}')
        print()


────────────────────────────────────────────────────────────────────────
  actions_usage  (58 pairs)
────────────────────────────────────────────────────────────────────────
Q: Show me how to use the `connect` action in ARO.
A: ```aro
(Application-Start: Demo) {
    Connect the <connection> to the <server-address>.
    Return an <OK: status> for the <result>.
}
```

Q: Show me how to use the `validate` action in ARO.
A: ```aro
(Application-Start: Demo) {
    Validate the <valid> for the <email-address>.
    Return an <OK: status> for the <result>.
}
```


────────────────────────────────────────────────────────────────────────
  actions_alias  (9 pairs)
────────────────────────────────────────────────────────────────────────
Q: How do I use `clear` in ARO? (alias for delete)
A: `Clear` is an alias for the `delete` action. Example:

```aro
(Application-Start: Demo) {
    Clear the <updated-cart> from the <item> in <cart>.
    Return an <OK: status> for the <result>.
}
```

Q: How do I 